In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
import os
import re
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

print("Libraries loaded successfully.")

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully.


In [3]:
dataset = load_dataset("goosmanlei/amazon_reviews_multi", "en")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
})


In [4]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

# Sample a manageable subset for aspect extraction
aspect_train_df = train_df.sample(
    n=10000,
    random_state=42
).reset_index(drop=True)

aspect_val_df = val_df.sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

aspect_test_df = test_df.sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

print("Train:", len(aspect_train_df))
print("Validation:", len(aspect_val_df))
print("Test:", len(aspect_test_df))

Train: 10000
Validation: 1000
Test: 1000


In [5]:
ASPECT_TERMS = {
    "delivery": [
        "delivery", "shipping", "shipment", "arrived",
        "arrival", "delivered", "deliver"
    ],
    "quality": [
        "quality", "durable", "durability", "material"
    ],
    "price": [
        "price", "cost", "value", "expensive", "cheap"
    ],
    "size": [
        "size", "sizing", "small", "large", "bigger",
        "smaller", "length", "width"
    ],
    "fit": [
        "fit", "fits", "fitting", "comfortable", "comfort"
    ],
    "battery": [
        "battery", "charging", "charge", "charger"
    ],
    "sound": [
        "sound", "audio", "volume", "bass", "noise"
    ],
    "customer_service": [
        "service", "support", "customer service"
    ],
    "packaging": [
        "packaging", "package", "box", "包装"
    ],
    "design": [
        "design", "style", "appearance", "look"
    ],
    "performance": [
        "performance", "works", "working", "function",
        "speed"
    ],
    "ease_of_use": [
        "easy", "easier", "simple", "use", "install",
        "installation"
    ],
}

print("Number of aspect categories:", len(ASPECT_TERMS))

for aspect, terms in ASPECT_TERMS.items():
    print(f"{aspect}: {len(terms)} terms")

Number of aspect categories: 12
delivery: 7 terms
quality: 4 terms
price: 5 terms
size: 8 terms
fit: 5 terms
battery: 4 terms
sound: 5 terms
customer_service: 3 terms
packaging: 4 terms
design: 4 terms
performance: 5 terms
ease_of_use: 6 terms


In [8]:
def create_aspect_labels(text, aspect_terms):
    tokens = re.findall(r"\b\w+\b", text.lower())
    labels = ["O"] * len(tokens)

    # Sort longer phrases first
    all_terms = []
    for aspect, terms in aspect_terms.items():
        for term in terms:
            term_tokens = term.lower().split()
            all_terms.append((term_tokens, aspect))

    all_terms.sort(key=lambda x: len(x[0]), reverse=True)

    for term_tokens, aspect in all_terms:
        term_len = len(term_tokens)

        for i in range(len(tokens) - term_len + 1):
            if tokens[i:i + term_len] == term_tokens:
                if all(label == "O" for label in labels[i:i + term_len]):
                    labels[i] = "B-ASPECT"

                    for j in range(1, term_len):
                        labels[i + j] = "I-ASPECT"

    return tokens, labels


# Test on one review
sample_text = aspect_train_df.loc[0, "review_body"]

tokens, labels = create_aspect_labels(
    sample_text,
    ASPECT_TERMS
)

for token, label in zip(tokens, labels):
    if label != "O":
        print(f"{token:20} -> {label}")

small                -> B-ASPECT


In [7]:
print("Review:")
print(sample_text)

print("\nDetected aspects:")

for token, label in zip(tokens, labels):
    if label != "O":
        print(f"{token:20} -> {label}")

Review:
Not strong enough to run a small 120v vacuum cleaner, to clean car.

Detected aspects:
small                -> B-ASPECT


In [9]:
def count_aspects(texts, aspect_terms):
    total_reviews = len(texts)
    reviews_with_aspects = 0
    total_aspects = 0

    for text in texts:
        _, labels = create_aspect_labels(text, aspect_terms)

        aspect_count = sum(
            1 for label in labels if label == "B-ASPECT"
        )

        if aspect_count > 0:
            reviews_with_aspects += 1
            total_aspects += aspect_count

    return total_reviews, reviews_with_aspects, total_aspects


total, with_aspects, total_aspects = count_aspects(
    aspect_train_df["review_body"].tolist(),
    ASPECT_TERMS
)

print("Total reviews:", total)
print("Reviews with detected aspects:", with_aspects)
print("Reviews without aspects:", total - with_aspects)
print("Total detected aspects:", total_aspects)
print("Average aspects per labeled review:",
      round(total_aspects / max(with_aspects, 1), 2))

Total reviews: 10000
Reviews with detected aspects: 5545
Reviews without aspects: 4455
Total detected aspects: 10747
Average aspects per labeled review: 1.94


In [10]:
def build_aspect_dataframe(df):
    rows = []

    for text in df["review_body"]:
        tokens, labels = create_aspect_labels(
            text,
            ASPECT_TERMS
        )

        rows.append({
            "tokens": tokens,
            "labels": labels
        })

    return pd.DataFrame(rows)


aspect_train_labeled = build_aspect_dataframe(aspect_train_df)
aspect_val_labeled = build_aspect_dataframe(aspect_val_df)
aspect_test_labeled = build_aspect_dataframe(aspect_test_df)

print("Train:", len(aspect_train_labeled))
print("Validation:", len(aspect_val_labeled))
print("Test:", len(aspect_test_labeled))

print("\nExample:")
print(aspect_train_labeled.iloc[0])

Train: 10000
Validation: 1000
Test: 1000

Example:
tokens    [not, strong, enough, to, run, a, small, 120v,...
labels       [O, O, O, O, O, O, B-ASPECT, O, O, O, O, O, O]
Name: 0, dtype: object


In [11]:
label2id = {
    "O": 0,
    "B-ASPECT": 1,
    "I-ASPECT": 2
}

id2label = {
    0: "O",
    1: "B-ASPECT",
    2: "I-ASPECT"
}

print("label2id:", label2id)
print("id2label:", id2label)

label2id: {'O': 0, 'B-ASPECT': 1, 'I-ASPECT': 2}
id2label: {0: 'O', 1: 'B-ASPECT', 2: 'I-ASPECT'}


In [12]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded successfully.
Vocabulary size: 30522


In [13]:
sample_tokens = aspect_train_labeled.iloc[0]["tokens"]
sample_labels = aspect_train_labeled.iloc[0]["labels"]

encoded = tokenizer(
    sample_tokens,
    is_split_into_words=True,
    truncation=True,
    max_length=128
)

word_ids = encoded.word_ids()

aligned_labels = []

previous_word_id = None

for word_id in word_ids:
    if word_id is None:
        aligned_labels.append(-100)

    elif word_id != previous_word_id:
        aligned_labels.append(
            label2id[sample_labels[word_id]]
        )

    else:
        # Same original word → continuation subword
        # receives the same label
        aligned_labels.append(
            label2id[sample_labels[word_id]]
        )

    previous_word_id = word_id


print("Tokens:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

print("\nWord IDs:")
print(word_ids)

print("\nAligned labels:")
print(aligned_labels)

print("\nLengths:")
print("Tokens:", len(encoded["input_ids"]))
print("Labels:", len(aligned_labels))

Tokens:
['[CLS]', 'not', 'strong', 'enough', 'to', 'run', 'a', 'small', '120', '##v', 'vacuum', 'cleaner', 'to', 'clean', 'car', '[SEP]']

Word IDs:
[None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 11, 12, None]

Aligned labels:
[-100, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, -100]

Lengths:
Tokens: 16
Labels: 16


In [14]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128
    )

    all_labels = []

    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        aligned_labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)

            elif word_id != previous_word_id:
                aligned_labels.append(
                    label2id[labels[word_id]]
                )

            else:
                aligned_labels.append(
                    label2id[labels[word_id]]
                )

            previous_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [15]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    aspect_train_labeled,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    aspect_val_labeled,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    aspect_test_labeled,
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['tokens', 'labels'],
    num_rows: 10000
})
Dataset({
    features: ['tokens', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['tokens', 'labels'],
    num_rows: 1000
})


In [16]:
tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=val_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=test_dataset.column_names
)

print("Train:", tokenized_train)
print("Validation:", tokenized_val)
print("Test:", tokenized_test)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 7416.55 examples/s]

Train: Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})
Validation: Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})
Test: Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})


In [17]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Model loaded successfully.")
print("Number of labels:", model.config.num_labels)
print("Labels:", model.config.id2label)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3780.09it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; no

Model loaded successfully.
Number of labels: 3
Labels: {0: 'O', 1: 'B-ASPECT', 2: 'I-ASPECT'}


In [18]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        current_predictions = []
        current_labels = []

        for pred, lab in zip(prediction, label):
            if lab != -100:
                current_predictions.append(pred)
                current_labels.append(lab)

        true_predictions.extend(current_predictions)
        true_labels.extend(current_labels)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        labels=[1, 2],
        average="micro",
        zero_division=0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Metrics function ready.")

Metrics function ready.


In [19]:
training_args = TrainingArguments(
    output_dir="./aspect_extraction_results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="steps",
    save_strategy="steps",

    eval_steps=500,
    save_steps=500,

    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,

    report_to="none",

    save_total_limit=2
)

print("Training arguments ready.")

Training arguments ready.


In [20]:
from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.0
)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[early_stopping]
)

print("Trainer created successfully.")
print("Early stopping patience:", 2)

Trainer created successfully.
Early stopping patience: 2


In [21]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("\nTrain samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_val))
print("Test samples:", len(tokenized_test))

print("\nLabels:", model.config.id2label)
print("Early stopping:", 2)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
VRAM: 7.96 GB

Train samples: 10000
Validation samples: 1000
Test samples: 1000

Labels: {0: 'O', 1: 'B-ASPECT', 2: 'I-ASPECT'}
Early stopping: 2


In [22]:
train_result = trainer.train()

print("\nTraining completed.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best F1:", trainer.state.best_metric)

Step,Training Loss,Validation Loss,Precision,Recall,F1
500,0.002818,0.001635,0.988701,0.997151,0.992908
1000,0.000606,0.001116,0.991525,1.000000,0.995745
1500,0.000241,0.000685,0.996212,0.999050,0.997629
1875,0.000533,0.000699,0.996212,0.999050,0.997629


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.51it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber


Training completed.
Best checkpoint: ./aspect_extraction_results\checkpoint-1500
Best F1: 0.9976292081555239


In [23]:
test_results = trainer.evaluate(
    tokenized_test
)

print("\nTest Results:")
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Training Loss,Validation Loss,Step,Precision,Recall,F1
0.000533,0.000387,1875,0.998079,1.000000,0.999038



Test Results:
eval_loss: 0.0004
eval_precision: 0.9981
eval_recall: 1.0000
eval_f1: 0.9990


In [24]:
import os

final_aspect_path = "../../models/aspects/bert_aspect_extraction"

os.makedirs(final_aspect_path, exist_ok=True)

trainer.save_model(final_aspect_path)
tokenizer.save_pretrained(final_aspect_path)

print("Aspect Extraction model saved successfully.")
print("Path:", final_aspect_path)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.32s/it]

Aspect Extraction model saved successfully.
Path: ../../models/aspects/bert_aspect_extraction


In [25]:
loaded_aspect_model = AutoModelForTokenClassification.from_pretrained(
    final_aspect_path
)

loaded_aspect_tokenizer = AutoTokenizer.from_pretrained(
    final_aspect_path
)

print("Saved model loaded successfully.")
print("Labels:", loaded_aspect_model.config.id2label)
print("Number of labels:", loaded_aspect_model.config.num_labels)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5092.78it/s]

Saved model loaded successfully.
Labels: {0: 'O', 1: 'B-ASPECT', 2: 'I-ASPECT'}
Number of labels: 3
